# 01 Power Curve Fitting
This notebook explains how SCADA-style turbine data is cleaned, preprocessed, binned, and converted into an empirical power curve. The sample data is synthetic and public-safe.


## Data Description
The sample SCADA table contains `timestamp`, `turbine_id`, `wind_speed`, `power_kw`, and `status`. In the private project these fields came from turbine-level reports; here they are synthetic examples with the same schema.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from wind_power_baselines.preprocessing import clean_scada
from wind_power_baselines.power_curve import build_power_curve
scada = pd.read_csv(PROJECT_ROOT / 'data_sample' / 'sample_turbine_scada.csv')
scada.head()


## Cleaning And EDA
We parse timestamps, keep normal rows, remove impossible wind speed and power values, add month and season labels, then inspect the cleaned wind-power scatter.


In [ ]:
cleaned = clean_scada(scada)
print({'raw_rows': len(scada), 'cleaned_rows': len(cleaned)})
sns.scatterplot(data=cleaned, x='wind_speed', y='power_kw', hue='turbine_id')


## Power Curve Construction
The curve uses wind-speed bins and median power. Median binning is more robust than fitting directly to raw scatter because turbine data may include curtailment, outage, and sensor artifacts.


In [ ]:
curve = build_power_curve(cleaned, bin_width=0.5, min_samples=2)
curve.to_csv(PROJECT_ROOT / 'data_sample' / 'sample_power_curve.csv', index=False)
sns.lineplot(data=curve, x='wind_speed_bin', y='power_kw', hue='turbine_id', marker='o')
curve
